In [1]:
import pandas as pd
import networkx as nx
import re
import yaml
from itertools import chain
from pathlib import Path
from operator import itemgetter
from collections import defaultdict
import json 
import requests
import matplotlib.pyplot as plt
import pickle

In [2]:
with open('indication_paths.yaml', 'r') as fh:
        ind = yaml.safe_load(fh)

In [3]:
def path_to_tup(path):
    return (path['graph']['drugbank'], path['graph']['disease_mesh'])

def path_to_G(path):
    return nx.node_link_graph(path)                                                                                        

def get_all_paths(path):
    source_id = path['links'][0]['source']                                                                              
    target_ids = list(set([l['target'] for l in path['links']]) - set([l['source'] for l in path['links']]))
    G = path_to_G(path)
    this_paths = list(chain(*[list(nx.all_simple_paths(G, source_id, target_id)) for target_id in target_ids]))         
    return this_paths

def get_id_to_type(G):
    id_to_type = {}
    for n in G.nodes.data():
        id_to_type[n[0]] = n[1]['label']
    return id_to_type

def get_id_to_name(G):
    id_to_name = {}
    for n in G.nodes.data():
        id_to_name[n[0]] = n[1]['name']
    return id_to_name

def add_metaedges(G):
    id_to_type = get_id_to_type(G)
    for e in G.edges:
        G.edges[e]['metaedge'] = id_to_type[e[0]] + ' - ' + e[2] + ' - ' + id_to_type[e[1]]
    return G

def add_meanode_pairs(G):
    id_to_type = get_id_to_type(G)
    for e in G.edges:
        G.edges[e]['mn_pair'] = id_to_type[e[0]] + ' - ' + id_to_type[e[1]]
    return G

def get_targets(G):
    drug = list(G.edges)[0][0]
    targets = []
    for e in G.edges:
        if e[0] == drug:
            targets.append(e[1])
    return targets

def get_target_metaedges(G):
    drug = list(G.edges)[0][0]
    target_mes = []
    if 'metaedge' not in G.edges[list(G.edges)[0]]:
        G = add_metaedges(G)
    
    for e in G.edges:
        if e[0] == drug:
            target_mes.append(G.edges[e]['metaedge'])
    return target_mes

In [4]:
basic_stats = defaultdict(list)
all_metaedges = []
all_parings = []
all_targets = []
unique_metaedges = []
first_edge_type = []
all_nodes = []

id_to_name = {}
id_to_label = {}

for i, p in enumerate(ind):
    drug_id, dis_id = path_to_tup(p)
    paths = get_all_paths(p)
    G = path_to_G(p)
    
    G = add_metaedges(G)
    G = add_meanode_pairs(G)
    
    basic_stats['idx'].append(i)
    basic_stats['id'].append(p['graph']['_id'])
    basic_stats['drug'].append(drug_id)
    basic_stats['disease'].append(dis_id)
    basic_stats['n_nodes'].append(len(G.nodes))
    basic_stats['n_edges'].append(len(G.edges))
    basic_stats['n_paths'].append(len(paths))
    basic_stats['longest_path'].append(max([len(p) for p in paths]))
    basic_stats['shortest_path'].append(max([len(p) for p in paths]))
    basic_stats['metapath'].append(" - ".join([n[1]['label'] for n in G.nodes.data()]))
    basic_stats['metapath_with_edges'].append("".join([re.sub(" - [^-]*$"," - ",e[2]['metaedge']) for e in G.edges.data()])+"Disease")

    
    this_metaedges = [G.edges[e]['metaedge'] for e in G.edges]
    all_metaedges += this_metaedges
    unique_metaedges += list(set(this_metaedges))
    
    all_parings += [G.edges[e]['mn_pair'] for e in G.edges]
    all_targets += get_targets(G)
    first_edge_type += get_target_metaedges(G)
    all_nodes += list(G.nodes)
    
    id_to_label = {**id_to_label, **get_id_to_type(G)}
    id_to_name = {**id_to_name, **get_id_to_name(G)}
    
basic_stats = pd.DataFrame(basic_stats)

In [5]:
basic_stats

,idx,id,drug,disease,n_nodes,n_edges,n_paths,longest_path,shortest_path,metapath,metapath_with_edges
0,0,DB00619_MESH_D015464_1,DB:DB00619,MESH:D015464,3,2,1,3,3,Drug - Protein - Disease,Drug - decreases activity of - Protein - cause...
1,1,DB00619_MESH_D034721_1,DB:DB00619,MESH:D034721,5,5,2,4,4,Drug - Protein - Protein - BiologicalProcess -...,Drug - decreases activity of - Drug - decrease...
2,2,DB00316_MESH_D010146_1,DB:DB00316,MESH:D010146,7,8,3,5,5,Drug - Protein - Protein - Protein - Biologica...,Drug - decreases activity of - Drug - decrease...
3,3,DB00316_MESH_D005334_1,DB:DB00316,MESH:D005334,5,4,1,5,5,Drug - Pathway - GrossAnatomicalStructure - Bi...,Drug - negatively regulates - Pathway - occurs...
4,4,DB00945_MESH_D010146_1,DB:DB00945,MESH:D010146,7,7,2,6,6,Drug - Protein - Protein - BiologicalProcess -...,Drug - decreases activity of - Drug - decrease...
...,...,...,...,...,...,...,...,...,...,...,...
4841,4841,DB01234_MESH_D009404_1,DB:DB01234,MESH:D009404,7,6,1,7,7,Drug - Protein - GeneFamily - GeneFamily - Bio...,Drug - positively regulates - Protein - increa...
4842,4842,DB01234_MESH_C562390_1,DB:DB01234,MESH:C562390,7,6,1,7,7,Drug - Protein - GeneFamily - GeneFamily - Bio...,Drug - positively regulates - Protein - increa...
4843,4843,DB01234_MESH_D000312_1,DB:DB01234,MESH:D000312,4,3,1,4,4,Drug - Protein - BiologicalProcess - Disease,Drug - positively regulates - Protein - negati...
4844,4844,DB01234_MESH_D000224_1,DB:DB01234,MESH:D000224,5,5,2,4,4,Drug - Protein - BiologicalProcess - Biologica...,Drug - positively regulates - Protein - positi...


In [6]:
large = basic_stats[(basic_stats['n_nodes'] <= 5) & (basic_stats['n_paths'] == 1)]

In [7]:
large

,idx,id,drug,disease,n_nodes,n_edges,n_paths,longest_path,shortest_path,metapath,metapath_with_edges
0,0,DB00619_MESH_D015464_1,DB:DB00619,MESH:D015464,3,2,1,3,3,Drug - Protein - Disease,Drug - decreases activity of - Protein - cause...
3,3,DB00316_MESH_D005334_1,DB:DB00316,MESH:D005334,5,4,1,5,5,Drug - Pathway - GrossAnatomicalStructure - Bi...,Drug - negatively regulates - Pathway - occurs...
6,6,DB00788_MESH_D010146_1,DB:DB00788,MESH:D010146,5,4,1,5,5,Drug - Protein - ChemicalSubstance - Biologica...,Drug - decreases activity of - Protein - incre...
9,9,DB01610_MESH_D003586_1,DB:DB01610,MESH:D003586,5,4,1,5,5,Drug - ChemicalSubstance - BiologicalProcess -...,Drug - increases abundance of - ChemicalSubsta...
10,10,DB00916_MESH_D018805_1,DB:DB00916,MESH:D018805,5,4,1,5,5,Drug - ChemicalSubstance - BiologicalProcess -...,Drug - increases abundance of - ChemicalSubsta...
...,...,...,...,...,...,...,...,...,...,...,...
4769,4769,DB01590_MESH_D014402_1,DB:DB01590,MESH:D014402,5,4,1,5,5,Protein - Drug - BiologicalProcess - Phenotypi...,Protein - positively regulates - Drug - decrea...
4773,4773,DB00819_MESH_D004832_1,DB:DB00819,MESH:D004832,4,3,1,4,4,Drug - Protein - ChemicalSubstance - Disease,Drug - decreases activity of - Protein - decre...
4780,4780,DB01193_MESH_D006973_1,DB:DB01193,MESH:D006973,5,4,1,5,5,Drug - Protein - ChemicalSubstance - Phenotypi...,Drug - decreases activity of - Protein - posit...
4843,4843,DB01234_MESH_D000312_1,DB:DB01234,MESH:D000312,4,3,1,4,4,Drug - Protein - BiologicalProcess - Disease,Drug - positively regulates - Protein - negati...


In [8]:
dmdb_nodes = {}
dmdb_mapping = {}

for index in large['idx']:
    for node in ind[index]['nodes']:
            
        if node['id'].startswith('DB:DB'):
            # Update the node['id'] by replacing 'UniProt' with 'UniProtKB'
            updated_id = node['id'].replace('DB', 'DrugBank:')
#             print('Updated ID: {}   Label: {}'.format(updated_id, node['label']))
            
            # Update the dmdb_nodes dictionary
            dmdb_nodes[updated_id] = node['label']
            dmdb_mapping[node['id']] = updated_id
            
        elif node['id'].startswith('DB:'):
            # Update the node['id'] by replacing 'UniProt' with 'UniProtKB'
            updated_id = node['id'].replace('DB:', 'DrugBank')
#             print('Updated ID: {}   Label: {}'.format(updated_id, node['label']))
            
            # Update the dmdb_nodes dictionary
            dmdb_nodes[updated_id] = node['label']
            dmdb_mapping[node['id']] = updated_id
        else:
            dmdb_nodes[node['id']] = node['label']
            dmdb_mapping[node['id']] = node['id']

In [9]:
mapping = {}
for i in dmdb_nodes: 
    # Handling MESH identifiers
    if i.split(':')[0] == 'MESH' and dmdb_nodes[i] == 'Drug': 
        result = requests.get('https://nodenormalization-sri.renci.org/get_normalized_nodes',
                              params={'curie': i})
        for j in result.json()[i]['equivalent_identifiers']: 
            if j['identifier'].startswith('DRUGBANK'): 
                mapping[i] = j['identifier'].lower().replace(':', '.')
                print(f"MESH Drug Mapped: {mapping[i]}")
                break
        else:
            mapping[i] = i.lower().replace(':', '.')

    elif i.split(':')[0] == 'MESH' and dmdb_nodes[i] == 'Disease': 
        result = requests.get('https://nodenormalization-sri.renci.org/get_normalized_nodes',
                              params={'curie': i})
        try: 
            for j in result.json()[i]['equivalent_identifiers']: 
                if j['identifier'].startswith('MONDO'): 
                    mapping[i] = j['identifier'].lower().replace(':', '.')
                    print(f"MESH Disease Mapped: {mapping[i]}")
                    break
            else:
                mapping[i] = i.lower().replace(':', '.')
        except: 
            continue
                
    elif i.split(':')[0] == 'MESH' and dmdb_nodes[i] == 'ChemicalSubstance': 
        result = requests.get('https://nodenormalization-sri.renci.org/get_normalized_nodes',
                              params={'curie': i})
        try: 
            for j in result.json()[i]['equivalent_identifiers']: 
                if j['identifier'].startswith('DRUGBANK'): 
                    mapping[i] = j['identifier'].lower().replace(':', '.')
                    print(f"ChemicalSubstance Mapped: {mapping[i]}")
                    break
            else:
                mapping[i] = i.lower().replace(':', '.')
        except: 
            continue

    elif i.split(':')[0] == 'MESH' and dmdb_nodes[i] == 'PhenotypicFeature': 
        result = requests.get('https://nodenormalization-sri.renci.org/get_normalized_nodes',
                              params={'curie': i})
        for j in result.json()[i]['equivalent_identifiers']: 
            if j['identifier'].startswith('MONDO'): 
                mapping[i] = j['identifier'].lower().replace(':', '.')
                print(f"PhenotypicFeature Mapped: {mapping[i]}")
                break
        else:
            mapping[i] = i.lower().replace(':', '.')
    
    elif i.split(':')[0] == 'UniProt': 
        mapping[i] = i.lower().replace(':', '.')
        print(f"UniProt Mapped: {mapping[i]}")

    elif i.split(':')[0] == 'HP': 
        # General handling of 'HP' identifiers
        print(f"Processing HP Identifier: {i}")
        if dmdb_nodes[i] == 'Disease':  # Specific HP + Disease condition
            result = requests.get('https://nodenormalization-sri.renci.org/get_normalized_nodes',
                                  params={'curie': i})
            for j in result.json()[i]['equivalent_identifiers']: 
                if j['identifier'].startswith('MONDO'): 
                    mapping[i] = j['identifier'].lower().replace(':', '.')
                    print(f"HP Disease Mapped to MONDO: {mapping[i]}")
                    break
            else:
                mapping[i] = i.lower().replace('hp:', 'hpo.')
        else:  # Generic HP condition
            print(f"Original HP Value: {i}")
            new_value = i.lower().replace('hp:', 'hpo.')
            print(f"Transformed HP Value: {new_value}")
            mapping[i] = new_value
            print(f"HP Mapped: {mapping[i]}")

    elif i.split(':')[0] == 'GO': 
        mapping[i] = i.lower().replace(':', '.')
        print(f"GO Mapped: {mapping[i]}")

    elif i.split(':')[0] == 'UBERON': 
        mapping[i] = i.lower().replace(':', '.')
        print(f"UBERON Mapped: {mapping[i]}")

    else: 
        mapping[i] = i.lower().replace(':', '.')
        print(f"Other Mapped: {mapping[i]}")


MESH Drug Mapped: drugbank.db00619
UniProt Mapped: uniprot.p00519
MESH Disease Mapped: mondo.0011996
MESH Drug Mapped: drugbank.db00316
Other Mapped: reactome.r-hsa-2162123
UBERON Mapped: uberon.0000955
GO Mapped: go.0001659
MESH Drug Mapped: drugbank.db00788
UniProt Mapped: uniprot.p35354
GO Mapped: go.0006954
MESH Drug Mapped: drugbank.db01610
ChemicalSubstance Mapped: drugbank.db01004
GO Mapped: go.0039693
Other Mapped: taxonomy.10358
MESH Disease Mapped: mondo.0005132
MESH Drug Mapped: drugbank.db00916
GO Mapped: go.0003676
Other Mapped: taxonomy.2
MESH Drug Mapped: drugbank.db01060
Other Mapped: tigr.02074
GO Mapped: go.0009252
Other Mapped: taxonomy.1314
MESH Disease Mapped: mondo.0001039
MESH Drug Mapped: drugbank.db00479
UniProt Mapped: uniprot.p0a7s3
GO Mapped: go.0006412
MESH Disease Mapped: mondo.0006670
MESH Drug Mapped: drugbank.db00438
Other Mapped: taxonomy.562
MESH Disease Mapped: mondo.0100338
MESH Drug Mapped: drugbank.db06762
UniProt Mapped: uniprot.q12791
GO Mapped:

In [10]:
modified_data = {key: '.'.join([part.split('.')[0], ''.join([x.upper() for x in part.split('.')[1]])]) for key, part in mapping.items()}

In [11]:
large_idx = list(large['idx'])

In [14]:
G = pickle.load(open('graph_v3.pkl', 'rb'))

In [15]:
large_idx = list(large['idx'])
for index in large['idx']:
    for node in ind[index]['nodes']:
        try: 
#             print(dmdb_mapping[node['id']])
#             print(modified_data[dmdb_mapping[node['id']]])
            if modified_data[dmdb_mapping[node['id']]] not in G.nodes: 
                print('Node not in G:', modified_data[dmdb_mapping[node['id']]])
                large_idx.remove(index)
                break
        except: 
            large_idx.remove(index)
            print('Node not in mapping:', node['id'])
            break


Node not in G: uberon.0000955
Node not in G: mesh.D011453
Node not in G: taxonomy.10358
Node not in G: mesh.C029371
Node not in G: tigr.02074
Node not in G: taxonomy.2
Node not in G: tigr.02074
Node not in G: taxonomy.562
Node not in G: mesh.D011453
Node not in G: mesh.C007852
Node not in G: mesh.C066928
Node not in G: mesh.D008595
Node not in G: taxonomy.485
Node not in G: uniprot.Q81VT3
Node not in G: mesh.D014812
Node not in G: tigr.02074
Node not in G: mesh.D015508
Node not in G: interpro.IPR028809
Node not in G: uniprot.P44350
Node not in G: taxonomy.1313
Node not in G: mesh.C106301
Node not in G: mesh.D015378
Node not in G: interpro.IPR028809
Node not in G: chebi.18111
Node not in G: taxonomy.727
Node not in G: mesh.D008595
Node not in G: mesh.D004967
Node not in mapping: MESH:D012338
Node not in G: cl.0000169
Node not in G: mesh.D013610
Node not in G: mesh.D014812
Node not in G: mesh.C039979
Node not in G: mesh.D010146
Node not in G: drugbank.DB05768
Node not in G: mesh.D004967


In [16]:
val_path = [ind[i] for i in large_idx]

In [17]:
path_list = []
for i in large_idx: 
    path_nodes = []
    for j in ind[i]['nodes']:
        path_nodes.append(modified_data[j['id']])
    if len(path_nodes) > 2:
        path_list.append(path_nodes)

In [18]:
len(path_list)

405

In [ ]:
# # Define the file path where you want to save the resulting list
# file_path = 'val_path_large.json'

# # Save the selected_dicts list to a JSON file
# with open(file_path, 'w') as file:
#     json.dump(val_path, file)